In [ ]:
from pyspark.sql import functions as F, types as T
from pyspark.sql import Window

df_lineitem     = spark.table("samples.tpch.lineitem")
df_transactions = spark.table("samples.bakehouse.sales_transactions")
df_trips        = spark.table("samples.nyctaxi.trips")

## Problem 1 - approx_count_distinct vs countDistinct

On `df_trips`, compare exact `countDistinct` vs `approx_count_distinct` for `pickup_zip`.

`approx_count_distinct` uses HyperLogLog and is faster on large data. Show both results side by side.

Also compute `F.approx_count_distinct(col, rsd=0.05)` with a custom relative standard deviation.

Expected columns: `exact_distinct`, `approx_distinct`

In [ ]:
# Test: both columns > 0, approx and exact values are close (within 10%)
row = df_distinct.collect()[0]
assert row["exact_distinct"] > 0, "exact_distinct should be > 0"
assert row["approx_distinct"] > 0, "approx_distinct should be > 0"
diff_ratio = abs(row["exact_distinct"] - row["approx_distinct"]) / row["exact_distinct"]
assert diff_ratio < 0.10, f"approx and exact differ by more than 10%: {diff_ratio:.2%}"
print("All tests passed for Problem 1")

## Problem 2 - stddev, variance, and mean

On `df_transactions` grouped by `franchiseID`, compute:

- `F.stddev("totalPrice")` - standard deviation
- `F.variance("totalPrice")` - variance
- `F.mean("totalPrice")` - mean (same as avg)
- `F.stddev_pop("totalPrice")` - population std dev
- `F.var_pop("totalPrice")` - population variance

Expected columns: `franchiseID`, `mean_price`, `std_price`, `var_price`, `std_pop`, `var_pop`

In [ ]:
# Test: std_price >= 0, var_price >= 0, mean_price > 0
for row in df_stats.collect():
    if row["std_price"] is not None:
        assert row["std_price"] >= 0, f"std_price should be >= 0, got {row['std_price']}"
    if row["var_price"] is not None:
        assert row["var_price"] >= 0, f"var_price should be >= 0, got {row['var_price']}"
    if row["mean_price"] is not None:
        assert row["mean_price"] > 0, f"mean_price should be > 0, got {row['mean_price']}"
print("All tests passed for Problem 2")

## Problem 3 - corr and covar_pop

On `df_lineitem`, compute the following across all rows:

- `F.corr("l_quantity", "l_extendedprice")` - Pearson correlation (-1 to 1)
- `F.covar_pop("l_quantity", "l_extendedprice")` - population covariance
- `F.covar_samp("l_quantity", "l_extendedprice")` - sample covariance

Expected: single-row DataFrame with columns `correlation_qty_price`, `covar_pop_qty_price`, `covar_samp_qty_price`

In [ ]:
# Test: correlation is between -1 and 1, covariance > 0 (price increases with quantity)
row = df_corr.collect()[0]
assert -1.0 <= row["correlation_qty_price"] <= 1.0, f"Correlation out of range: {row['correlation_qty_price']}"
assert row["covar_pop_qty_price"] > 0, "Covariance should be positive (price increases with quantity)"
assert row["covar_samp_qty_price"] > 0, "Sample covariance should be positive"
print("All tests passed for Problem 3")

## Problem 4 - percentile_approx and median

On `df_trips`, compute approximate percentiles for `fare_amount`:

- `F.percentile_approx("fare_amount", 0.5)` - median (50th percentile)
- `F.percentile_approx("fare_amount", 0.25)` - 25th percentile
- `F.percentile_approx("fare_amount", 0.75)` - 75th percentile
- `F.percentile_approx("fare_amount", [0.1, 0.5, 0.9])` - multiple percentiles at once as array

Expected columns: `p25`, `p50`, `p75`, `decile_array`

In [ ]:
# Test: p25 <= p50 <= p75, decile_array has 3 elements
row = df_percentiles.collect()[0]
assert row["p25"] <= row["p50"], f"p25 should be <= p50: {row['p25']} vs {row['p50']}"
assert row["p50"] <= row["p75"], f"p50 should be <= p75: {row['p50']} vs {row['p75']}"
assert len(row["decile_array"]) == 3, f"decile_array should have 3 elements, got {len(row['decile_array'])}"
print("All tests passed for Problem 4")

## Problem 5 - skewness and kurtosis

On `df_trips`, compute skewness and kurtosis of both `fare_amount` and `trip_distance` distributions.

These measure distribution shape: skewness captures asymmetry and kurtosis captures tail heaviness.

Build a two-row DataFrame (one per column) with columns: `column_name`, `skewness_val`, `kurtosis_val`

In [ ]:
# Test: skewness and kurtosis are numeric (not null)
rows = df_shape.collect()
assert len(rows) == 2, f"Expected 2 rows, got {len(rows)}"
for row in rows:
    assert row["skewness_val"] is not None, f"skewness_val is null for {row['column_name']}"
    assert row["kurtosis_val"] is not None, f"kurtosis_val is null for {row['column_name']}"
print("All tests passed for Problem 5")

## Problem 6 - rollup (subtotals)

Use `.rollup()` on `df_transactions` to compute revenue at three levels:

1. Overall total (`franchiseID=null`, `product=null`)
2. Per franchise total (`product=null`)
3. Per franchise-product total

```python
df_transactions.rollup("franchiseID", "product").agg(
    F.sum("totalPrice").alias("revenue"),
    F.count("*").alias("txn_count")
)
```

Expected columns: `franchiseID`, `product`, `revenue`, `txn_count`

In [ ]:
# Test: result includes rows where both franchiseID and product are null (grand total)
# count is > number of franchise-product combos
grand_total_rows = df_rollup.filter(
    F.col("franchiseID").isNull() & F.col("product").isNull()
).count()
assert grand_total_rows == 1, f"Expected exactly 1 grand total row, got {grand_total_rows}"

plain_count = df_transactions.groupBy("franchiseID", "product").count().count()
rollup_count = df_rollup.count()
assert rollup_count > plain_count, (
    f"Rollup rows ({rollup_count}) should exceed plain groupBy rows ({plain_count})"
)
print("All tests passed for Problem 6")

## Problem 7 - cube (all combinations)

Use `.cube()` on `df_transactions` with columns `franchiseID`, `product`, `paymentMethod`.

This generates subtotals for ALL combinations: grand total, per `franchiseID`, per `product`, per `paymentMethod`, all two-way combinations, and the full three-way breakdown.

Use `F.grouping_id()` to identify which level each row represents.

Expected columns: `franchiseID`, `product`, `paymentMethod`, `revenue`, `grouping_level`

In [ ]:
# Test: result has more rows than a plain groupBy, includes null rows (subtotals)
plain_count = df_transactions.groupBy("franchiseID", "product", "paymentMethod").count().count()
cube_count = df_cube.count()
assert cube_count > plain_count, (
    f"Cube rows ({cube_count}) should exceed plain groupBy rows ({plain_count})"
)

null_rows = df_cube.filter(
    F.col("franchiseID").isNull() & F.col("product").isNull() & F.col("paymentMethod").isNull()
).count()
assert null_rows == 1, f"Expected 1 grand total row with all nulls, got {null_rows}"
print("All tests passed for Problem 7")

## Problem 8 - collect_list for ranked aggregation

Use `F.collect_list` and `F.sort_array` together to build an ordered list of transaction totals per franchise, then use `F.slice` to get the top 5.

Also compute `F.array_min` and `F.array_max` on the collected sorted list.

Expected columns: `franchiseID`, `all_prices_sorted`, `top_5_prices`, `min_price`, `max_price`

In [ ]:
# Test: top_5_prices has at most 5 elements, min_price <= max_price
for row in df_collected.collect():
    assert len(row["top_5_prices"]) <= 5, (
        f"top_5_prices has more than 5 elements for franchiseID={row['franchiseID']}"
    )
    if row["min_price"] is not None and row["max_price"] is not None:
        assert row["min_price"] <= row["max_price"], (
            f"min_price ({row['min_price']}) > max_price ({row['max_price']}) "
            f"for franchiseID={row['franchiseID']}"
        )
print("All tests passed for Problem 8")